In this demo we will go over the basics of the RayJobClient in the SDK

In [ ]:
# Import pieces from codeflare-sdk
from codeflare_sdk import Cluster, ClusterConfiguration, RayJobClient, set_api_client
from kube_authkit import AuthConfig, get_k8s_client

In [ ]:
# Authenticate to your Kubernetes/OpenShift cluster using kube-authkit

# Option 1: Auto-detect credentials (kubeconfig or in-cluster service account)
# NOTE: In RHOAI Workbenches the workbench service account may not have Ray RBAC
# permissions. Use Option 2 (token) unless your admin has granted SA permissions
# (see RHOAIENG-46748). Auto-detect works if you have a local kubeconfig.
# auth_config = AuthConfig(method="auto")

# Option 2 (Recommended for RHOAI Workbenches): Token-based authentication
# Get your token with: oc whoami -t  (or from the OpenShift console → Copy login command)
# Note: auth_token is also used below as the bearer token for the RayJobClient
auth_token = "sha256~XXXXX"  # oc whoami -t
auth_config = AuthConfig(
    method="openshift",
    k8s_api_host="https://api.example.com:6443",
    token=auth_token,
)

# Option 3: OIDC authentication (for BYOIDC-enabled clusters)
# auth_config = AuthConfig(
#     method="oidc",
#     k8s_api_host="https://api.example.com:6443",
#     oidc_issuer="https://your-oidc-provider.com",
#     client_id="your-client-id",
#     use_device_flow=True,
# )
# auth_token = ...  # Retrieve access token from your OIDC provider for RayJobClient

api_client = get_k8s_client(config=auth_config)
set_api_client(api_client)


NOTE: The default images used by the CodeFlare SDK for creating a RayCluster resource depend on the installed Python version:

- For Python 3.11: 'quay.io/modh/ray:2.52.1-py311-cu121'
- For Python 3.12: 'quay.io/modh/ray:2.53.0-py312-cu128'

If you prefer to use a custom Ray image that better suits your needs, you can specify it in the image field to override the default.

In [ ]:
# Create and configure our cluster object
# The SDK will try to find the name of your default local queue based on the annotation "kueue.x-k8s.io/default-queue": "true" unless you specify the local queue manually below
cluster = Cluster(ClusterConfiguration(
    name='jobtest',
    head_memory_requests=6,
    head_memory_limits=8,
    head_extended_resource_requests={'nvidia.com/gpu':0}, # For GPU enabled workloads set the head_extended_resource_requests and worker_extended_resource_requests
    worker_extended_resource_requests={'nvidia.com/gpu':0},
    num_workers=2,
    worker_cpu_requests=1,
    worker_cpu_limits=1,
    worker_memory_requests=4,
    worker_memory_limits=6,
    # image="", # Optional Field 
    write_to_file=False # When enabled Ray Cluster yaml files are written to /HOME/.codeflare/resources 
    # local_queue="local-queue-name" # Specify the local queue manually
))

In [ ]:
# Bring up the cluster
cluster.apply()
cluster.wait_ready()

In [ ]:
cluster.details()

### Ray Job Submission - Authorized Ray Cluster

* Submit a job using an authorized Ray dashboard and the Job Submission Client
* Provide an entrypoint command directed to your job script
* Set up your runtime environment

In [ ]:
# Gather the dashboard URL
ray_dashboard = cluster.cluster_dashboard_uri()

# Create the header for passing your bearer token
header = {
    'Authorization': f'Bearer {auth_token}'
}

# Initialize the RayJobClient
client = RayJobClient(address=ray_dashboard, headers=header, verify=True)

In [ ]:
# Submit an example mnist job using the RayJobClient
submission_id = client.submit_job(
    entrypoint="python mnist.py",
    runtime_env={"working_dir": "./","pip": "requirements.txt"},
)
print(submission_id)

In [ ]:
# Get the job's logs
client.get_job_logs(submission_id)

In [ ]:
# Get the job's status
client.get_job_status(submission_id)

In [ ]:
# Get job related info
client.get_job_info(submission_id)

In [ ]:
# List all existing jobs
client.list_jobs()

In [ ]:
# Iterate through the logs of a job 
async for lines in client.tail_job_logs(submission_id):
    print(lines, end="") 

In [ ]:
# Delete a job
# Can run client.cancel_job(submission_id) first if job is still running
client.delete_job(submission_id)

### Unauthorized Ray Cluster with the Ray Job Client

In [ ]:
"""
Initialise the RayJobClient with the Ray Dashboard
"""
ray_dashboard = cluster.cluster_dashboard_uri()
client = RayJobClient(address=ray_dashboard, verify=False)

In [ ]:
# Submit an example mnist job using the RayJobClient
submission_id = client.submit_job(
    entrypoint="python mnist.py",
    runtime_env={"working_dir": "./","pip": "requirements.txt"},
)
print(submission_id)

In [ ]:
# Stop the job 
client.stop_job(submission_id)

In [ ]:
# Delete the job
client.delete_job(submission_id)

In [ ]:
cluster.down()

In [ ]:
# No explicit logout needed - authentication is managed automatically by kube-authkit